In [ ]:
!pip install langchain-groq


In [ ]:
from google.colab import userdata
groq_api_key=userdata.get("GROQ_API_KEY")
from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.7,
)

In [ ]:
llm.invoke("I want to know the meaning of water ").content

# Pydantic Schema

In [ ]:
from pydantic import BaseModel,Field

class  graph_schema(BaseModel):
  topic: str=Field(description="The topic of the graph")
  post:str= Field(description="The Linkedin post content")
  #curated post for updated information for LLM
  curated_post: str= Field(description="The Curated Linkedin post content")
#parse the object

demo_obj = graph_schema(
    topic="The importance of data privacy in the digital age",
    post="Data privacy is crucial in today's digital world. With the increasing amount of personal information being shared online, it's important to understand how to protect your data and maintain your privacy.",
    curated_post="In the digital age, data privacy is more important than ever. With the"
)
print(demo_obj)

#


# Create the Nodes

In [ ]:
def create_post(state:graph_schema)-> graph_schema:
  #convert pydantic model to dictionary
  state=state.model_dump()
  #Extract the topic from the state
  topic=state['topic']
  #passing the topic to the LLm to generetate a linkedin post
  post=llm.invoke(f"write a linkedin post about {topic}").content

  # update the state with the generated post
  state['post']=post

  return state

def curate_post(state:graph_schema)-> graph_schema:
  #convert the pydantic model to a dictionary
  state=state.model_dump()
  # Extract the post from the state
  post=state['post']

  #passing the post to the LLM to curate it
  curated_post=llm.invoke(f"Curate the following linkedin post with human tone and make it perfect : {post}").content

  #update the state with the curated post
  state['curated_post']=curated_post

  return state


In [ ]:
from langgraph.graph import StateGraph,START,END

graph=StateGraph(graph_schema)
graph.add_node("create_post",create_post)
graph.add_node("curate_post",curate_post)

graph.add_edge(START,"create_post")
graph.add_edge("create_post","curate_post")
graph.add_edge("curate_post",END)

#compile the graph
pydantic_graph=graph.compile()

from IPython.display import Image,display
Image(pydantic_graph.get_graph().draw_mermaid_png())

In [ ]:
pydantic_graph.invoke(
    {
        "topic":"The importance of data privacy in the digital age",
        "post":"",
        "curated_post":""
    }
)